# D0 — Playground: tiny graphs to build intuition / Patio de juegos: grafos pequeños para la intuición

🇬🇧 Before the story starts, let's play. This notebook gives you a one-line helper to **draw any small graph,
count its paths, and see where information piles up**. Edit the edge lists and re-run — everything updates.
Come back here whenever a later notebook feels abstract.

🇪🇸 Antes de empezar la historia, juguemos. Este cuaderno te da una función de una línea para **dibujar
cualquier grafo pequeño, contar sus caminos y ver dónde se acumula la información**. Edita las listas de
aristas y vuelve a ejecutar — todo se actualiza. Vuelve aquí cuando un cuaderno posterior se sienta abstracto.

In [ ]:
import numpy as np, networkx as nx, matplotlib.pyplot as plt
from aiq.quiver import Quiver
from aiq.path_algebra import PathAlgebra
from oversquash.ideal_bridge import walk_operators

def explore(edges, n_nodes, src=0, dst=None, max_len=4, pos=None):
    """Draw a graph and report path counts src->dst at each length.
    edges: list of (from, to). / Dibuja un grafo y reporta conteos de caminos."""
    dst = n_nodes - 1 if dst is None else dst
    G = nx.DiGraph(); G.add_nodes_from(range(n_nodes)); G.add_edges_from(edges)
    pos = pos or nx.spring_layout(G, seed=1)
    cols = ['#34d399' if i==dst else '#60a5fa' if i==src else '#cbd5e1' for i in range(n_nodes)]
    plt.figure(figsize=(5.5,3))
    nx.draw(G, pos, node_color=cols, with_labels=True, node_size=600, arrowsize=14, edge_color='#94a3b8')
    plt.title(f'{n_nodes} nodes — src={src} (blue) → dst={dst} (green)'); plt.show()
    ei = np.array(edges).T
    raw, eff = walk_operators(ei, n_nodes, max_length=max_len)
    print(f'paths {src} → {dst}:')
    for g in range(max_len):
        r = int(raw[g][src,dst]); e = int(eff[g][src,dst])
        extra = '' if r==e else f'   (kQ/I de-duplicates → {e})'
        if r: print(f'  length {g+1}: {r} path(s){extra}')
    return raw, eff

## Toy 1 — a straight line (no redundancy) / Una línea recta (sin redundancia)

🇬🇧 The simplest long-range graph: `0→1→2→3→4`. There is exactly **one** path from end to end. Distance is
large, but there is *no pile-up* — so a normal GNN can (slowly) carry the signal. Over-squashing needs
**many** competing paths, which a line doesn't have.

🇪🇸 El grafo de largo alcance más simple: `0→1→2→3→4`. Hay exactamente **un** camino de punta a punta. La
distancia es grande, pero *no hay acumulación* — así que una GNN normal puede (lentamente) llevar la señal. El
over-squashing necesita **muchos** caminos en competencia, que una línea no tiene.

In [ ]:
line = [(0,1),(1,2),(2,3),(3,4)]
pos = {i:(i,0) for i in range(5)}
_ = explore(line, 5, src=0, dst=4, pos=pos)

## Toy 2 — two routes of different length / Dos rutas de distinta longitud

🇬🇧 A short edge `0→3` and a long detour `0→1→2→3`. Information from `0` reaches `3` at **two different
ranges** (length 1 and length 3). This is why we index everything by `g`: different hop-counts are different
channels.

🇪🇸 Una arista corta `0→3` y un desvío largo `0→1→2→3`. La información de `0` llega a `3` en **dos rangos
distintos** (longitud 1 y longitud 3). Por esto indexamos todo por `g`: distintos números de saltos son
canales distintos.

In [ ]:
two_routes = [(0,3),(0,1),(1,2),(2,3)]
pos = {0:(0,0), 1:(1,1), 2:(2,1), 3:(3,0)}
_ = explore(two_routes, 4, src=0, dst=3, pos=pos)

## Toy 3 — parallel routes of the SAME length (redundancy) / Rutas paralelas de la MISMA longitud (redundancia)

🇬🇧 The 'diamond': `0→1→3` and `0→2→3`, both length 2. Two paths, but if the middle nodes carry the *same*
content they are **redundant**. Watch `kQ/I` collapse the count from 2 to 1 — this is the de-duplication the
later notebooks discuss (and show is *not* the right fix for learning).

🇪🇸 El 'diamante': `0→1→3` y `0→2→3`, ambos de longitud 2. Dos caminos, pero si los nodos del medio llevan el
*mismo* contenido son **redundantes**. Mira cómo `kQ/I` colapsa el conteo de 2 a 1 — esta es la
de-duplicación que los cuadernos siguientes discuten (y muestran que *no* es el arreglo correcto para aprender).

In [ ]:
diamond = [(0,1),(0,2),(1,3),(2,3)]
pos = {0:(0,0), 1:(1,1), 2:(1,-1), 3:(2,0)}
_ = explore(diamond, 4, src=0, dst=3, pos=pos)

## Now you / Ahora tú

🇬🇧 Change the edges below and re-run. Try: a 'double diamond' (two diamonds in a row) to get **4** paths;
or add a third parallel branch to get **3**. The more parallel same-length paths, the worse the squashing.

🇪🇸 Cambia las aristas de abajo y vuelve a ejecutar. Prueba: un 'doble diamante' (dos diamantes en fila) para
obtener **4** caminos; o añade una tercera rama paralela para obtener **3**. Cuantos más caminos paralelos de
la misma longitud, peor el apretujamiento.

In [ ]:
# double diamond: 0 -> {1,2} -> 3 -> {4,5} -> 6   (4 paths 0->6 of length 4)
my_graph = [(0,1),(0,2),(1,3),(2,3),(3,4),(3,5),(4,6),(5,6)]
_ = explore(my_graph, 7, src=0, dst=6, max_len=4)

🇬🇧 **Ready?** Now go to `D1_what_is_oversquashing.ipynb` to see why all this path-counting matters for real
networks. / 🇪🇸 **¿Listo?** Ahora ve a `D1_what_is_oversquashing.ipynb` para ver por qué todo este conteo de
caminos importa para redes reales.